In [1]:
import os
os.chdir(os.path.dirname(os.path.abspath("rag_vector.ipynb")))

## RAG with Vector Search

We swap keyword search for vector search in the RAG pipeline. Only the `search` step changes — `build_prompt` and `llm` are inherited unchanged from `RAGBase`.

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [5]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

"Yes, you can still sign up for the course. However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted."

Now build the embeddings and vector index so we can swap in vector search.

In [6]:
texts = []

for doc in documents:
    text = doc['question'] + " " + doc['answer']
    texts.append(text)

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
from tqdm.auto import tqdm

vectors = []
batch_size = 50

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    vector = model.encode(batch)
    vectors.extend(vector)

  0%|          | 0/27 [00:00<?, ?it/s]

In [9]:
import numpy as np

X = np.array(vectors)

In [10]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

### RAGVector

Subclass `RAGBase`, override `search` to encode the query and use the vector index instead of the text index.

In [11]:
from rag_helper import RAGVector

vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [12]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still sign up for the course even if it has already begun. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start learning and submitting homework without registering, as registration is mainly for gauging interest.'

### Using sqlitesearch for persistent vector search

In [13]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

vs_index.fit(X, documents)

In [14]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [15]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [16]:
vs_index.close()